# 🔧 [F 재실행] Few-shot + CoT 베이스라인 — max_new_tokens 버그 수정판

**버그였던 것**: 이전 노트북(`tombench_tier3_experiments.ipynb`)의 `evaluate()`는 `max_new_tokens=12`로
고정되어 있었습니다. zero-shot/포맷전용(C3)처럼 모델이 바로 `[[X]]`만 내뱉는 조건에서는 문제없지만,
F(few-shot+CoT)는 예시에 `"Let's reason step by step: ..."` 같은 추론 문장을 포함시켰기 때문에,
모델이 추론을 시작하자마자 12토큰에서 잘려서 답 자체를 못 냈습니다 (실제 raw 출력의 84.7%가 `[[`
패턴조차 없었음). 그 결과가 ToMi −36pp, SocialIQa −32pp 같은 비정상적 하락이었습니다.

**수정 내용**: F 조건에서만 `max_new_tokens=200`으로 늘렸습니다. (D/A/C3는 이미 정상이었으므로 이 노트북에서는 F만 다시 돕니다.)

**독립 실행 가능**: 이전 세션을 안 열어도 됩니다. 새 Colab에서 이 노트북만 위에서부터 실행하면 됩니다.


## ⚙️ [1/5] 환경 설정

In [ ]:
import sys, os, subprocess, time
import torch
if not torch.cuda.is_available():
    print('\n❌ GPU 미설정. 런타임 → 런타임 유형 변경 → GPU'); raise SystemExit
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')

print('\n📦 라이브러리 설치 (2~3분)...')
t0 = time.time()
!pip install -q unsloth
!pip install -q "datasets<4.0.0" scikit-learn pandas tqdm
print(f'   ⏱ {time.time()-t0:.0f}초')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = 'ToMBench_연구결과'
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
TIER3_DIR = f'{DRIVE_DIR}/tier3_F_fix'   # 이전 결과와 섞이지 않도록 새 폴더
os.makedirs(TIER3_DIR, exist_ok=True)
print(f'✅ Drive: {DRIVE_DIR}  |  출력: {TIER3_DIR}')


## 📥 [2/5] ToMBench 로드 + 분할 (기존과 완전히 동일한 코드 — 손대지 마세요)

In [ ]:
import json, glob, math, re, random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TOMBENCH_DATA_PATHS = [
    '/content/ToMBench/data', '/content/data',
    '/content/drive/MyDrive/ToMBench-main/data',
    '/content/drive/MyDrive/ToMBench/data',
]
OPENTOM_DATA_PATHS = [
    '/content/OpenToM/data',
    '/content/drive/MyDrive/OpenToM-main/data',
    '/content/drive/MyDrive/OpenToM/data',
]
TEST_RATIO = 0.30
VAL_RATIO_OF_TRAIN = 0.20
EVAL_MAX_PER_DATASET = 1000

TOMBENCH_DIR = None
for p in TOMBENCH_DATA_PATHS:
    if os.path.isdir(p) and glob.glob(os.path.join(p, '*.jsonl')):
        TOMBENCH_DIR = p; break
if TOMBENCH_DIR is None:
    subprocess.run(['git','clone','--depth','1','https://github.com/zhchen18/ToMBench.git','/content/ToMBench'], check=False)
    TOMBENCH_DIR = '/content/ToMBench/data'
assert os.path.isdir(TOMBENCH_DIR)
print(f'✅ ToMBench: {TOMBENCH_DIR}')

def is_valid(x):
    if x is None: return False
    if isinstance(x, float) and math.isnan(x): return False
    s = str(x).strip()
    return len(s) > 0 and s.lower() != 'nan'

def normalize_ability(ab):
    return ab.replace('Non-literal', 'Non-Literal')

def build_user_prompt(story, question, opts):
    lines = [f'{l}. {t}' for l, t in opts]
    return (f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines))

records = []
for fp in sorted(glob.glob(os.path.join(TOMBENCH_DIR, '*.jsonl'))):
    task = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            ability = normalize_ability(row.get('能力\nABILITY', 'Unknown'))
            category = ability.split(':')[0].strip()
            ans = row.get('答案\nANSWER') or row.get('ANSWER')
            if not is_valid(row.get('STORY')) or not is_valid(row.get('QUESTION')) or not is_valid(ans): continue
            opts = []
            for letter in ['A','B','C','D']:
                v = row.get(f'OPTION-{letter}')
                if is_valid(v): opts.append((letter, str(v).strip()))
            records.append({
                'source': 'ToMBench', 'task': task, 'category': category, 'ability': ability,
                'user_prompt': build_user_prompt(row['STORY'], row['QUESTION'], opts),
                'answer': str(ans).strip().upper()[0],
            })
df = pd.DataFrame(records)

SMALL_THRESHOLD = 30
ab_counts = df['ability'].value_counts()
small_ab = ab_counts[ab_counts < SMALL_THRESHOLD].index.tolist()
large_ab = ab_counts[ab_counts >= SMALL_THRESHOLD].index.tolist()
large_df = df[df['ability'].isin(large_ab)].reset_index(drop=True)
train_val_l, test_l = train_test_split(large_df, test_size=TEST_RATIO, random_state=42, stratify=large_df['ability'])
train_l, val_l = train_test_split(train_val_l, test_size=VAL_RATIO_OF_TRAIN, random_state=42, stratify=train_val_l['ability'])
parts = {'train': [], 'val': [], 'test': []}
for ab in small_ab:
    ab_df = df[df['ability'] == ab].reset_index(drop=True)
    n = len(ab_df)
    if n < 5:
        parts['train'].append(ab_df); continue
    seed_ab = 42 + (hash(ab) % 100)
    tr, vt = train_test_split(ab_df, test_size=0.4, random_state=seed_ab)
    va, te = train_test_split(vt, test_size=0.5, random_state=seed_ab)
    parts['train'].append(tr); parts['val'].append(va); parts['test'].append(te)
train_s = pd.concat(parts['train']) if parts['train'] else pd.DataFrame()
val_s = pd.concat(parts['val']) if parts['val'] else pd.DataFrame()
test_s = pd.concat(parts['test']) if parts['test'] else pd.DataFrame()
train_df = pd.concat([train_l, train_s], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([val_l, val_s],     ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([test_l, test_s],   ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}  (1607/411/842 이어야 함)')
assert len(train_df)==1607 and len(val_df)==411 and len(test_df)==842


## 🤖 [3/5] 모델 로드 + 평가 헬퍼 (배치 처리 버전, max_new_tokens 넉넉하게)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
EVAL_BATCH_SIZE = 8  # OOM 나면 4로

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)

ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')
def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt, few_shot_prefix=None):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}]
    if few_shot_prefix:
        msgs += few_shot_prefix
    msgs.append({'role':'user','content':user_prompt})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval', few_shot_prefix=None,
             batch_size=EVAL_BATCH_SIZE, max_new_tokens=12):
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    results = []
    for i in tqdm(range(0, len(eval_records), batch_size), desc=desc):
        batch = eval_records[i:i+batch_size]
        prompts = [to_text(tokenizer, r['user_prompt'], few_shot_prefix=few_shot_prefix) for r in batch]
        inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True,
                            max_length=MAX_SEQ_LENGTH).to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, inputs['input_ids'].shape[1]:]
        gens = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        for r, gen in zip(batch, gens):
            pred = extract_letter(gen)
            results.append({**r, 'pred': pred, 'raw': gen, 'correct': int(pred == r['answer'])})
    return pd.DataFrame(results)

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    return model, tokenizer

print('✅ 헬퍼 준비 완료 (배치 처리 + max_new_tokens 조절 가능)')


## 📦 [4/5] 외부 벤치마크 로더 (기존과 동일)

In [ ]:
def cap_eval(recs, n=EVAL_MAX_PER_DATASET):
    if n and len(recs) > n: return random.Random(42).sample(recs, n)
    return recs

def load_opentom():
    OPENTOM_DIR = None
    for p in OPENTOM_DATA_PATHS:
        if os.path.isdir(p): OPENTOM_DIR = p; break
    if OPENTOM_DIR is None:
        subprocess.run(['git','clone','--depth','1','https://github.com/seacowx/OpenToM.git','/content/OpenToM'], check=False)
        if os.path.isdir('/content/OpenToM/data'): OPENTOM_DIR = '/content/OpenToM/data'
    if OPENTOM_DIR is None: print('⚠️ OpenToM 없음'); return []
    meta_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'meta_data.json')
    recs = []
    if os.path.exists(meta_path):
        with open(meta_path) as f: meta = json.load(f)
        att_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'attitude.json')
        if os.path.exists(att_path):
            with open(att_path) as f: att = json.load(f)
            opts_text = ['positive','negative','neutral']
            for sid, qs in att.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip().lower()
                    if gold not in opts_text: continue
                    shuffled = opts_text[:]
                    random.Random((hash(sid+q.get('question','')) & 0xffffffff)).shuffle(shuffled)
                    gold_letter = 'ABC'[shuffled.index(gold)]
                    recs.append({
                        'source':'OpenToM','task':'attitude','category':'Emotion/Attitude','ability':'Attitude',
                        'user_prompt':build_user_prompt(narrative, q['question'], list(zip('ABC',shuffled))),
                        'answer':gold_letter,
                    })
        cg_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'location_cg_fo.json')
        if os.path.exists(cg_path):
            with open(cg_path) as f: cg = json.load(f)
            for sid, qs in cg.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip()
                    if gold not in ('Yes','No'): continue
                    opts = [('A','Yes'),('B','No')]
                    gold_letter = 'A' if gold == 'Yes' else 'B'
                    recs.append({
                        'source':'OpenToM','task':'location-cg','category':'Belief','ability':'Location False Beliefs',
                        'user_prompt': build_user_prompt(narrative, q['question'], opts),
                        'answer': gold_letter,
                    })
    return cap_eval(recs)

def load_tomi():
    TOMI_DIR = '/content/ToMi'
    if not os.path.isdir(TOMI_DIR):
        r = subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/ToMi.git', TOMI_DIR], capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 클론 실패'); return []
    DATA_OUT = '/content/tomi_data'
    os.makedirs(DATA_OUT, exist_ok=True)
    if not os.path.exists(os.path.join(DATA_OUT,'test.txt')):
        r = subprocess.run([sys.executable,'main.py','-n','500','-o',DATA_OUT,'-s','42'], cwd=TOMI_DIR, capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 생성 실패'); return []
    samples = []; cur_facts, cur_qas = [], []; prev = 0
    with open(os.path.join(DATA_OUT,'test.txt')) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip(): continue
            m = re.match(r'^(\d+)\s(.*)$', line)
            if not m: continue
            num = int(m.group(1)); rest = m.group(2)
            if num <= prev and (cur_facts or cur_qas):
                narr = ' '.join(cur_facts)
                for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
                cur_facts, cur_qas = [], []
            prev = num
            if '\t' in rest:
                parts = rest.split('\t')
                if len(parts) >= 2: cur_qas.append((parts[0].strip(), parts[1].strip()))
            else:
                cur_facts.append(rest.strip())
        if cur_facts or cur_qas:
            narr = ' '.join(cur_facts)
            for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
    ans_pool = list({s['answer'] for s in samples})
    if len(ans_pool) < 4: return []
    recs = []
    for s in samples:
        gold = s['answer']
        distractors = [a for a in ans_pool if a != gold]
        ds = random.Random(hash(s['narrative'][:40]) & 0xffffffff).sample(distractors, 3)
        opts = [gold] + ds
        random.Random(hash(s['question']) & 0xffffffff).shuffle(opts)
        gold_letter = 'ABCD'[opts.index(gold)]
        recs.append({
            'source':'ToMi','task':'location-belief','category':'Belief','ability':'Location False Beliefs',
            'user_prompt': build_user_prompt(s['narrative'], s['question'], list(zip('ABCD', opts))),
            'answer': gold_letter,
        })
    return cap_eval(recs)

def load_socialiqa():
    from datasets import load_dataset
    ds = None
    for name in ['allenai/social_i_qa','social_i_qa']:
        try:
            ds = load_dataset(name, split='validation'); print(f'✅ SocialIQa from HF: {name}'); break
        except Exception:
            try:
                ds = load_dataset(name, split='validation', trust_remote_code=True); print(f'✅ SocialIQa from HF (legacy): {name}'); break
            except Exception as e2:
                print(f'  ↳ {name}: {type(e2).__name__}'); continue
    if ds is None:
        try:
            import urllib.request, zipfile, glob as _glob
            url = 'https://storage.googleapis.com/ai2-mosaic/public/socialiqa/socialiqa-train-dev.zip'
            zip_path = '/content/socialiqa.zip'
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as z: z.extractall('/content/socialiqa_data')
            dev_jsonl = _glob.glob('/content/socialiqa_data/**/dev.jsonl', recursive=True)
            dev_lbl   = _glob.glob('/content/socialiqa_data/**/dev-labels.lst', recursive=True)
            if dev_jsonl and dev_lbl:
                with open(dev_jsonl[0]) as f: rows = [json.loads(l) for l in f]
                with open(dev_lbl[0]) as f: lbls = [l.strip() for l in f]
                ds = [dict(r, label=lbls[i]) for i,r in enumerate(rows)]
                print(f'✅ SocialIQa from AllenAI zip: {len(ds)}개')
        except Exception as e:
            print(f'  ↳ AllenAI zip 실패: {e}')
    if ds is None: print('⚠️ SocialIQa 로드 완전 실패 (스킵)'); return []
    recs = []
    for x in ds:
        ctx = x.get('context',''); q = x.get('question','')
        a,b,c = x.get('answerA',''), x.get('answerB',''), x.get('answerC','')
        lbl = str(x.get('label','')).strip()
        if lbl not in ('1','2','3'): continue
        gold_letter = {'1':'A','2':'B','3':'C'}[lbl]
        recs.append({
            'source':'SocialIQa','task':'social_cs','category':'Mixed','ability':'Mixed',
            'user_prompt': build_user_prompt(ctx, q, [('A',a),('B',b),('C',c)]),
            'answer': gold_letter,
        })
    return cap_eval(recs)

import re as _re_hitom
def _parse_hitom_choices(choices_str):
    if not isinstance(choices_str, str): return []
    pat = _re_hitom.compile(r'([A-Z])\.\s*([^,]+?)(?=,\s*[A-Z]\.|$)')
    return [(m.group(1).upper(), m.group(2).strip()) for m in pat.finditer(choices_str)]

def load_hitom():
    from datasets import load_dataset
    ds = None
    for name in ['Hi-ToM/Hi-ToM_Dataset', 'umwyf/Hi-ToM_Dataset']:
        try:
            ds = load_dataset(name, split='train'); print(f'✅ HiToM from {name}'); break
        except Exception as e:
            print(f'  ↳ {name}: {type(e).__name__}')
    if ds is None:
        try:
            HITOM_DIR = '/content/Hi-ToM_dataset'
            if not os.path.isdir(HITOM_DIR):
                subprocess.run(['git','clone','--depth','1','https://github.com/ying-hui-he/Hi-ToM_dataset.git', HITOM_DIR], capture_output=True, check=False)
            import glob as _glob
            json_files = _glob.glob(f'{HITOM_DIR}/**/*.json', recursive=True) + _glob.glob(f'{HITOM_DIR}/**/*.jsonl', recursive=True)
            if json_files:
                rows = []
                for fp in json_files:
                    with open(fp) as f:
                        try: data = json.load(f); rows.extend(data if isinstance(data, list) else [data])
                        except Exception: f.seek(0); rows.extend(json.loads(l) for l in f if l.strip())
                if rows: ds = rows; print(f'✅ HiToM from GitHub clone: {len(rows)}개')
        except Exception as e:
            print(f'  ↳ GitHub clone 실패: {e}')
    if ds is None: print('⚠️ HiToM 자동 로드 실패 (스킵)'); return []
    recs = []
    skipped = 0
    LETTERS = 'ABCDEFGHIJKLMNO'
    for x in ds:
        story = x.get('story') or x.get('context') or x.get('narrative') or ''
        q = x.get('question') or ''
        choices_raw = x.get('choices') or x.get('options') or []
        gold_raw = x.get('answer')
        if gold_raw is None: gold_raw = x.get('label')
        if not story or not q or not choices_raw or gold_raw is None: skipped += 1; continue
        if isinstance(choices_raw, str):
            opts = _parse_hitom_choices(choices_raw)
            if not opts:
                try:
                    parsed = json.loads(choices_raw); opts = list(zip(LETTERS, parsed))
                except Exception:
                    parts = [p.strip() for p in choices_raw.split('|')]; opts = list(zip(LETTERS, parts))
        elif isinstance(choices_raw, list):
            opts = list(zip(LETTERS, choices_raw))
        else:
            skipped += 1; continue
        if not opts: skipped += 1; continue
        gold_str = str(gold_raw).strip()
        gold_letter = None
        if len(gold_str) == 1 and gold_str.upper() in LETTERS:
            gold_letter = gold_str.upper()
        else:
            for letter, text in opts:
                if text.lower() == gold_str.lower(): gold_letter = letter; break
            if gold_letter is None:
                for letter, text in opts:
                    if gold_str.lower() in text.lower() or text.lower() in gold_str.lower(): gold_letter = letter; break
        if gold_letter is None: skipped += 1; continue
        if len(opts) > 5:
            gold_opt = next(o for o in opts if o[0] == gold_letter)
            others = [o for o in opts if o[0] != gold_letter]
            sampled = random.Random(hash(story+q) & 0xffffffff).sample(others, min(4, len(others)))
            new_pool = [gold_opt] + sampled
            random.Random(hash(q) & 0xffffffff).shuffle(new_pool)
            opts = [('ABCDE'[i], t) for i,(_, t) in enumerate(new_pool)]
            for i,(_, t) in enumerate(new_pool):
                if t == gold_opt[1]: gold_letter = 'ABCDE'[i]; break
        order = x.get('question_order', None)
        ability_label = f'High-Order False Beliefs (order={order})' if order is not None else 'High-Order False Beliefs'
        recs.append({
            'source':'HiToM','task':'high-order','category':'Belief','ability':ability_label,
            'user_prompt': build_user_prompt(story, q, opts),
            'answer': gold_letter,
        })
    if skipped: print(f'  ↳ HiToM: {skipped}개 스킵')
    return cap_eval(recs)

eval_sets = {'ToMBench': test_df.to_dict('records')}
eval_sets['OpenToM'] = load_opentom()
eval_sets['ToMi'] = load_tomi()
eval_sets['SocialIQa'] = load_socialiqa()
eval_sets['HiToM'] = load_hitom()
eval_sets = {k:v for k,v in eval_sets.items() if len(v) > 0}
print('📌 평가 예정 데이터셋:', {k: len(v) for k,v in eval_sets.items()})


## 🎯 [5/5] F 재실행 — CoT few-shot, max_new_tokens=200 (수정판)

이전과 같은 3개 예시를 쓰되, 이번엔 모델이 추론을 다 마치고 `[[X]]`까지 낼 수 있도록 여유를 줍니다.

In [ ]:
BASE_ACC = {'ToMBench': 61.52, 'OpenToM': 63.10, 'ToMi': 75.30, 'SocialIQa': 67.40, 'HiToM': 63.00}
FEWSHOT_IDX = [0, 1, 2]

def build_cot_fewshot(tokenizer):
    exemplars = train_df.iloc[FEWSHOT_IDX]
    msgs = []
    for _, r in exemplars.iterrows():
        reasoning = (f"Let's reason step by step: I track who has access to which information in the story, "
                     f"then check what the relevant character would infer given only what they observed. "
                     f"This points to option {r['answer']}.")
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"{reasoning} [[{r['answer']}]]"})
    return msgs

print('='*60); print('🎯 [F 수정판] Few-shot + CoT 베이스라인 평가 (max_new_tokens=200)'); print('='*60)
base_model, base_tokenizer = load_fresh_model()
fewshot_cot = build_cot_fewshot(base_tokenizer)

fewshot_results = {}
for name, recs in eval_sets.items():
    res_df = evaluate(recs, base_model, base_tokenizer, desc=f'F-fix-{name}',
                       few_shot_prefix=fewshot_cot, max_new_tokens=200)
    res_df.to_csv(f'{TIER3_DIR}/F_fewshot_cot_{name}.csv', index=False)
    acc = res_df['correct'].mean()*100
    has_bracket = res_df['raw'].astype(str).str.contains(r'\[\[').mean()*100
    fewshot_results[name] = acc
    print(f'  {name}: zero-shot base={BASE_ACC.get(name, float("nan")):.2f}%  →  few-shot+CoT={acc:.2f}%  '
          f'(Δ={acc-BASE_ACC.get(name,0):+.2f}pp)  [[X]]패턴 포함 비율={has_bracket:.1f}%')

with open(f'{TIER3_DIR}/F_fewshot_cot_summary.json','w') as f:
    json.dump(fewshot_results, f, indent=2, ensure_ascii=False)
print(f'\n💾 저장 완료: {TIER3_DIR}/')
print('   [[X]] 패턴 포함 비율이 90% 이상이면 정상, 여전히 낮으면 max_new_tokens를 더 늘려야 합니다.')
